
# B3c — Analysis: Downsampled-Weather Advantage Decomposition

**Purpose.** Combine `b3c_panda_predictions.csv` and
`b3c_chronos_predictions.csv` (produced by the two model-side
notebooks, run in their separate environments) to compute:

1. Panda-vs-Chronos advantage in each of the three conditions
   (`native_H96`, `hourly_H96_fixedsample`, `hourly_H16_fixedphys`),
   reported both **including and excluding degenerate channel-windows**
   (near-zero-variance channels, e.g. `rain (mm)` during a dry window --
   see below).
2. **Advantage decomposition** (Section 1.2 policy): for each horizon
   convention, does the advantage move because Panda's MAE moved
   (**H-i, structure hypothesis**) or because Chronos's MAE moved
   (**H-ii, frequency-affinity confound**)?
3. Wilcoxon significance per condition, relative skill
   (MAE_Chronos/MAE_Panda), and a plain-text auto-computed verdict.
4. An optional per-channel structure-vs-advantage correlation, if you
   supply the Experiment 30 per-channel structure statistic as a CSV.

**On the `degenerate` column:** both model-side notebooks guard
near-zero-variance channel-windows (dividing by a near-zero std
otherwise produces near-infinite normalized error) and flag which
(condition, window, channel) triples were guarded. This notebook
reports summaries both ways rather than silently picking one, the same
way Experiment 30 excluded near-constant channels from its own summary
rather than letting them dominate it -- a channel with zero variance
for an entire 512-step context (e.g. no rain for ~3.5 days) is close to
intrinsically unforecastable in a meaningful sense, so whether to
include it is a real judgment call worth seeing explicitly, not one to
make silently.

This notebook does not itself run any model -- pure CSV analysis,
consistent with keeping model inference isolated to their own pinned
environments.


In [1]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon

panda_df = pd.read_csv('b3c_panda_predictions.csv')
chronos_df = pd.read_csv('b3c_chronos_predictions.csv')

print('Panda rows:', len(panda_df), '| Chronos rows:', len(chronos_df))
print('Conditions (Panda):', sorted(panda_df.condition.unique()))
print('Conditions (Chronos):', sorted(chronos_df.condition.unique()))


Panda rows: 1260 | Chronos rows: 1260
Conditions (Panda): ['hourly_H16_fixedphys', 'hourly_H96_fixedsample', 'native_H96']
Conditions (Chronos): ['hourly_H16_fixedphys', 'hourly_H96_fixedsample', 'native_H96']


## Per-window MAE (averaged across channels) and the paired advantage test

Averaging across channels first (per window), then treating each
window as one paired observation for Wilcoxon -- matches this
project's standard evaluation protocol (Section 1.2) rather than
treating (window, channel) pairs as independent draws, which would
overstate the effective sample size.

Computed twice: **all** (every channel) and **clean** (excluding any
channel-window either model flagged as degenerate). Compare the two --
if they tell substantially different stories, that itself is worth
reporting rather than picking whichever one is more convenient.

In [2]:
def per_window_mae(df, exclude_degenerate=False):
    d = df[~df['degenerate']] if exclude_degenerate else df
    return d.groupby(['condition', 'window_idx'])['mae'].mean().reset_index()

def compute_summary(exclude_degenerate):
    panda_win = per_window_mae(panda_df, exclude_degenerate).rename(columns={'mae': 'panda_mae'})
    chronos_win = per_window_mae(chronos_df, exclude_degenerate).rename(columns={'mae': 'chronos_mae'})
    merged = panda_win.merge(chronos_win, on=['condition', 'window_idx'], how='inner')
    merged['advantage'] = merged['chronos_mae'] - merged['panda_mae']

    rows = []
    for condition, sub in merged.groupby('condition'):
        panda_mae = sub['panda_mae'].mean()
        chronos_mae = sub['chronos_mae'].mean()
        adv = chronos_mae - panda_mae
        rel_skill = chronos_mae / panda_mae if panda_mae > 0 else np.nan
        try:
            stat, p = wilcoxon(sub['chronos_mae'], sub['panda_mae'], alternative='greater')
        except ValueError:
            p = np.nan
        rows.append({
            'condition': condition, 'n_windows': len(sub),
            'panda_mae': panda_mae, 'chronos_mae': chronos_mae,
            'advantage': adv, 'relative_skill': rel_skill, 'wilcoxon_p': p,
        })
    out = pd.DataFrame(rows).set_index('condition')
    return out.reindex(['native_H96', 'hourly_H96_fixedsample', 'hourly_H16_fixedphys']), merged

summary_df, merged_all = compute_summary(exclude_degenerate=False)
summary_clean_df, merged_clean = compute_summary(exclude_degenerate=True)

n_deg_panda = panda_df['degenerate'].sum()
n_deg_chronos = chronos_df['degenerate'].sum()
print(f'Degenerate channel-windows flagged: Panda={n_deg_panda}, Chronos={n_deg_chronos}')
print()
print('=== ALL channels ===')
print(summary_df.round(4))
print()
print('=== EXCLUDING degenerate channel-windows ===')
print(summary_clean_df.round(4))

summary_df.to_csv('b3c_summary_all.csv')
summary_clean_df.to_csv('b3c_summary_clean.csv')


Degenerate channel-windows flagged: Panda=8, Chronos=8

=== ALL channels ===
                        n_windows  panda_mae  chronos_mae  advantage  \
condition                                                              
native_H96                     20     0.7654       0.8410     0.0756   
hourly_H96_fixedsample         20     0.8080       0.7620    -0.0461   
hourly_H16_fixedphys           20     0.4962       0.4657    -0.0305   

                        relative_skill  wilcoxon_p  
condition                                           
native_H96                      1.0988      0.0884  
hourly_H96_fixedsample          0.9430      0.9053  
hourly_H16_fixedphys            0.9385      0.9430  

=== EXCLUDING degenerate channel-windows ===
                        n_windows  panda_mae  chronos_mae  advantage  \
condition                                                              
native_H96                     20     0.7819       0.8566     0.0747   
hourly_H96_fixedsample         20  

## Reference check against Experiment 8

If `native_H96` doesn't land close to Experiment 8's Weather H=96
numbers (Panda 0.6378, Chronos 0.8115, advantage +0.1534), something
upstream (checkpoint loading, window construction, normalisation) needs
fixing before the downsampling comparison below can be trusted.

In [3]:
ref = summary_df.loc['native_H96']
ref_clean = summary_clean_df.loc['native_H96']
print(f"This run,   native_H96 (all):   Panda={ref.panda_mae:.4f}, Chronos={ref.chronos_mae:.4f}, "
      f"advantage={ref.advantage:+.4f}")
print(f"This run,   native_H96 (clean): Panda={ref_clean.panda_mae:.4f}, Chronos={ref_clean.chronos_mae:.4f}, "
      f"advantage={ref_clean.advantage:+.4f}")
print(f"Exp. 8 ref, Weather H96:        Panda=0.6378, Chronos=0.8115, advantage=+0.1534")


This run,   native_H96 (all):   Panda=0.7654, Chronos=0.8410, advantage=+0.0756
This run,   native_H96 (clean): Panda=0.7819, Chronos=0.8566, advantage=+0.0747
Exp. 8 ref, Weather H96:        Panda=0.6378, Chronos=0.8115, advantage=+0.1534


## Advantage decomposition (the decisive step)

For each horizon convention, decompose the native-to-hourly movement
into the Panda-side change and the Chronos-side change separately.
**Do not read the advantage number alone** -- the whole point of this
design is that a shrinking advantage is ambiguous between H-i and H-ii
until decomposed.

Run on the **degenerate-excluded** summary as the primary read (the
guarded-but-included near-zero-variance channels are edge cases, not
representative dynamics); the all-channels version is also shown so you
can see whether they disagree.

In [4]:
def decompose(summary, condition_native, condition_hourly, label):
    row_native = summary.loc[condition_native]
    row_hourly = summary.loc[condition_hourly]

    delta_panda = row_hourly.panda_mae - row_native.panda_mae
    delta_chronos = row_hourly.chronos_mae - row_native.chronos_mae
    delta_advantage = row_hourly.advantage - row_native.advantage

    print(f'--- {label} ---')
    print(f'  Advantage: {row_native.advantage:+.4f} (native) -> {row_hourly.advantage:+.4f} (hourly), '
          f'delta = {delta_advantage:+.4f}')
    print(f'  Panda MAE:   {row_native.panda_mae:.4f} -> {row_hourly.panda_mae:.4f}, delta = {delta_panda:+.4f}')
    print(f'  Chronos MAE: {row_native.chronos_mae:.4f} -> {row_hourly.chronos_mae:.4f}, delta = {delta_chronos:+.4f}')

    if abs(delta_panda) < 1e-6 and abs(delta_chronos) < 1e-6:
        verdict = 'NEITHER MODEL MOVED -- advantage stable, no support for H-i or H-ii'
    elif abs(delta_panda) > 2 * abs(delta_chronos):
        verdict = 'H-i dominant (Panda-side change >> Chronos-side change)' if delta_panda > 0 else \
                  'Panda IMPROVED at hourly res -- opposite of H-i, unexpected, inspect closely'
    elif abs(delta_chronos) > 2 * abs(delta_panda):
        verdict = 'H-ii dominant (Chronos-side change >> Panda-side change)' if delta_chronos < 0 else \
                  'Chronos WORSENED at hourly res -- opposite of H-ii, unexpected, inspect closely'
    else:
        verdict = 'MIXED -- both sides moved by comparable magnitude, neither H-i nor H-ii dominates cleanly'
    print(f'  Auto-read: {verdict}')
    print()
    return {'label': label, 'delta_panda': delta_panda, 'delta_chronos': delta_chronos,
            'delta_advantage': delta_advantage, 'verdict': verdict}

print('##### PRIMARY (degenerate-excluded) #####\n')
decomp_results = []
decomp_results.append(decompose(summary_clean_df, 'native_H96', 'hourly_H96_fixedsample',
                                 'Fixed sample-horizon (H=96 both res)'))
decomp_results.append(decompose(summary_clean_df, 'native_H96', 'hourly_H16_fixedphys',
                                 'Fixed physical-horizon (16h both res)'))
decomp_df = pd.DataFrame(decomp_results)
decomp_df.to_csv('b3c_decomposition_clean.csv', index=False)

print('##### COMPARISON (all channels, for reference) #####\n')
decomp_results_all = []
decomp_results_all.append(decompose(summary_df, 'native_H96', 'hourly_H96_fixedsample',
                                     'Fixed sample-horizon (H=96 both res)'))
decomp_results_all.append(decompose(summary_df, 'native_H96', 'hourly_H16_fixedphys',
                                     'Fixed physical-horizon (16h both res)'))
decomp_df_all = pd.DataFrame(decomp_results_all)
decomp_df_all.to_csv('b3c_decomposition_all.csv', index=False)


##### PRIMARY (degenerate-excluded) #####

--- Fixed sample-horizon (H=96 both res) ---
  Advantage: +0.0747 (native) -> -0.0461 (hourly), delta = -0.1208
  Panda MAE:   0.7819 -> 0.8080, delta = +0.0261
  Chronos MAE: 0.8566 -> 0.7620, delta = -0.0947
  Auto-read: H-ii dominant (Chronos-side change >> Panda-side change)

--- Fixed physical-horizon (16h both res) ---
  Advantage: +0.0747 (native) -> -0.0305 (hourly), delta = -0.1052
  Panda MAE:   0.7819 -> 0.4962, delta = -0.2857
  Chronos MAE: 0.8566 -> 0.4657, delta = -0.3910
  Auto-read: MIXED -- both sides moved by comparable magnitude, neither H-i nor H-ii dominates cleanly

##### COMPARISON (all channels, for reference) #####

--- Fixed sample-horizon (H=96 both res) ---
  Advantage: +0.0756 (native) -> -0.0461 (hourly), delta = -0.1217
  Panda MAE:   0.7654 -> 0.8080, delta = +0.0426
  Chronos MAE: 0.8410 -> 0.7620, delta = -0.0790
  Auto-read: MIXED -- both sides moved by comparable magnitude, neither H-i nor H-ii dominates cl

## Paired significance test: does each model's own MAE actually move?

Everything above tests Panda-vs-Chronos *within* a condition. This is
the missing piece: for each model separately, is its native-vs-hourly
MAE change (same windows, paired by `window_idx`) actually
distinguishable from noise, or within the range you'd expect from
random window-to-window variation? A magnitude-based "H-i dominant"
read from the decomposition cell above is a heuristic, not a
significance claim -- this closes that gap. Two-sided test, since
there's no strong pre-registered directional prior on which way either
model should move.

In [5]:
def paired_native_vs_hourly_test(df, condition_native, condition_hourly, exclude_degenerate=True):
    d = df[~df['degenerate']] if exclude_degenerate else df
    win_mae = d.groupby(['condition', 'window_idx'])['mae'].mean().reset_index()
    native = win_mae[win_mae.condition == condition_native].set_index('window_idx')['mae']
    hourly = win_mae[win_mae.condition == condition_hourly].set_index('window_idx')['mae']
    common = native.index.intersection(hourly.index)
    native, hourly = native.loc[common], hourly.loc[common]
    diff = hourly - native
    try:
        stat, p = wilcoxon(diff)  # two-sided by default
    except ValueError:
        p = np.nan
    direction = 'hourly WORSE' if diff.median() > 0 else ('hourly BETTER' if diff.median() < 0 else 'no change')
    return {
        'n_windows': len(common), 'native_median': native.median(), 'hourly_median': hourly.median(),
        'delta_median': diff.median(), 'wilcoxon_p': p, 'direction': direction,
    }

print('=== Paired native-vs-hourly test, per model (degenerate-excluded) ===\n')
paired_rows = []
for model_name, df in [('panda', panda_df), ('chronos', chronos_df)]:
    for hourly_cond, label in [('hourly_H96_fixedsample', 'Fixed sample-horizon'),
                                 ('hourly_H16_fixedphys', 'Fixed physical-horizon')]:
        res = paired_native_vs_hourly_test(df, 'native_H96', hourly_cond)
        res['model'], res['convention'] = model_name, label
        paired_rows.append(res)
        sig = '*' if res['wilcoxon_p'] < 0.05 else ' '
        print(f"{model_name:8s} | {label:24s} | n={res['n_windows']:2d} | "
              f"native={res['native_median']:.4f} hourly={res['hourly_median']:.4f} "
              f"delta={res['delta_median']:+.4f} p={res['wilcoxon_p']:.4f}{sig} ({res['direction']})")

paired_df = pd.DataFrame(paired_rows)
paired_df.to_csv('b3c_paired_significance.csv', index=False)
print()
print('* = p < 0.05. H-i requires: Panda direction = \'hourly WORSE\' AND significant, Chronos not (or weaker).')


=== Paired native-vs-hourly test, per model (degenerate-excluded) ===

panda    | Fixed sample-horizon     | n=20 | native=0.7531 hourly=0.7185 delta=-0.0196 p=0.5459  (hourly BETTER)
panda    | Fixed physical-horizon   | n=20 | native=0.7531 hourly=0.4225 delta=-0.1811 p=0.0000* (hourly BETTER)
chronos  | Fixed sample-horizon     | n=20 | native=0.7696 hourly=0.6477 delta=-0.0578 p=0.1769  (hourly BETTER)
chronos  | Fixed physical-horizon   | n=20 | native=0.7696 hourly=0.3838 delta=-0.3418 p=0.0000* (hourly BETTER)

* = p < 0.05. H-i requires: Panda direction = 'hourly WORSE' AND significant, Chronos not (or weaker).


## Robustness check: do both horizon conventions agree?

If the two conventions above point to different verdicts, that
disagreement is itself informative (it means one convention's
particular confound -- physical-horizon-length in the fixed-sample
case, or short-horizon favoritism in the fixed-physical case -- may be
driving the result, rather than the resolution change itself).

In [6]:
v1, v2 = decomp_df.iloc[0]['verdict'], decomp_df.iloc[1]['verdict']
same_direction = ('H-i' in v1) == ('H-i' in v2) and ('H-ii' in v1) == ('H-ii' in v2)
print('Fixed-sample-horizon verdict:  ', v1)
print('Fixed-physical-horizon verdict:', v2)
print()
if same_direction:
    print('Both conventions agree -- stronger evidence for whichever hypothesis they both support.')
else:
    print('Conventions DISAGREE -- read this as inconclusive rather than picking whichever '
          'convention supports a preferred story. Report both, flag the disagreement plainly.')


Fixed-sample-horizon verdict:   H-ii dominant (Chronos-side change >> Panda-side change)
Fixed-physical-horizon verdict: MIXED -- both sides moved by comparable magnitude, neither H-i nor H-ii dominates cleanly

Both conventions agree -- stronger evidence for whichever hypothesis they both support.


## Optional: per-channel structure-vs-advantage correlation

If you have the Experiment 30 per-channel structure statistic
(`top1/floor`) saved as a CSV with columns `channel, top1_floor`, this
cell correlates it against per-channel advantage at hourly resolution --
a secondary check on whether channels the structure statistic called
"high-structure" also show larger Panda advantage once sampling rate is
controlled. Experiment 33 already found this null at native resolution
(rho=+0.11, p=0.64); this reruns the same check at hourly resolution.
Skip if you don't have this file -- it's secondary, not required for
the main H-i/H-ii result above.

In [7]:
import os

structure_path = 'weather_structure_stats.csv'  # channel, top1_floor -- adjust path if needed
if os.path.exists(structure_path):
    structure_df = pd.read_csv(structure_path)

    panda_hourly = panda_df[panda_df.condition == 'hourly_H96_fixedsample']
    chronos_hourly = chronos_df[chronos_df.condition == 'hourly_H96_fixedsample']
    panda_by_ch = panda_hourly.groupby('channel')['mae'].mean()
    chronos_by_ch = chronos_hourly.groupby('channel')['mae'].mean()
    per_channel_adv = (chronos_by_ch - panda_by_ch).rename('advantage').reset_index()

    merged_struct = per_channel_adv.merge(structure_df, on='channel', how='inner')
    from scipy.stats import spearmanr
    rho, p = spearmanr(merged_struct['top1_floor'], merged_struct['advantage'])
    print(f'Structure-vs-advantage correlation at hourly resolution: rho={rho:.3f}, p={p:.3f}')
    print('Experiment 33 reference (native resolution): rho=+0.11, p=0.64')
else:
    print(f'{structure_path} not found -- skipping the optional structure-correlation check. '
          'This does not affect the main H-i/H-ii result above.')


weather_structure_stats.csv not found -- skipping the optional structure-correlation check. This does not affect the main H-i/H-ii result above.


## Summary

Print the full picture in one place before writing this up as a log
entry: the three-condition MAE/advantage table, the decomposition
verdicts, and the cross-convention agreement check.

In [8]:
print('=== Condition summary (ALL channels) ===')
print(summary_df.round(4))
print()
print('=== Condition summary (EXCLUDING degenerate) ===')
print(summary_clean_df.round(4))
print()
print('=== Decomposition (degenerate-excluded, primary) ===')
print(decomp_df[['label', 'delta_panda', 'delta_chronos', 'delta_advantage', 'verdict']].to_string(index=False))


=== Condition summary (ALL channels) ===
                        n_windows  panda_mae  chronos_mae  advantage  \
condition                                                              
native_H96                     20     0.7654       0.8410     0.0756   
hourly_H96_fixedsample         20     0.8080       0.7620    -0.0461   
hourly_H16_fixedphys           20     0.4962       0.4657    -0.0305   

                        relative_skill  wilcoxon_p  
condition                                           
native_H96                      1.0988      0.0884  
hourly_H96_fixedsample          0.9430      0.9053  
hourly_H16_fixedphys            0.9385      0.9430  

=== Condition summary (EXCLUDING degenerate) ===
                        n_windows  panda_mae  chronos_mae  advantage  \
condition                                                              
native_H96                     20     0.7819       0.8566     0.0747   
hourly_H96_fixedsample         20     0.8080       0.7620    -0.046